In [1]:
import pandas as pd

# Load
covariates = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")
correction = pd.read_csv("Input_phase_2/Correction_crm_users.csv")
upload = pd.read_csv("Input_phase_2/treatment_selected_multi_2.csv")

# Merge upload with covariates
df = upload.merge(covariates, on="customer_nk", how="inner")

# Flag customers present in correction file
corrected_ids = correction["user_id"].drop_duplicates()
df["in_correction"] = df["customer_nk"].isin(corrected_ids).astype(int)

C:\Users\tsterk\AppData\Local\Temp\ipykernel_2800\789347368.py:4: DtypeWarning: Columns (0: total_volume, 1: food_total, 2: sports_total, 3: monetary_value_52wk, 4: online_sales_52w, 5: retail_sales_52w, 6: monetary_value_53w_104w, 7: online_sales_53w_104w, 8: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  covariates = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")


In [2]:
df.head()

,customer_nk,original_incentive_name,has_rfl,gender,country_sk,recency,frequency,monetary_value,total_volume,length_of_relationship,...,monetary_value_52wk,volume_52wk,online_sales_52w,retail_sales_52w,frequency_53w_104w,monetary_value_53w_104w,volume_53w_104w,online_sales_53w_104w,retail_sales_53w_104w,in_correction
0,efe78c42-8e88-4bcc-8ffa-cfa578b05450,BNLX_ChurnP_SKUe_test_export.csv,1,F,hbi|eu|nl,549,1,30.27,2,549,...,30.27,2,0,30.27,0,0,0,0.0,0,1
1,9bd7ba92-c610-4a0a-bc99-fadc258c6be7,BNLX_ChurnP_250_test_export.csv,1,F,hbi|eu|nl,"1,673",1,38.52,1,"1,673",...,0,0,0.0,0,0,0,0,0,0,1
2,7b0f0c77-4f68-4871-9e74-7d268a3c7938,BNLX_ChurnP_10_test_export.csv,1,NaN,hbi|eu|be,366,1,23.6,5,366,...,23.6,5,0,23.6,0,0,0,0,0,1
3,fa785469-47ae-4984-9aef-32d7b42003c4,BNLX_ChurnP_10_test_export.csv,1,NaN,hbi|eu|be,371,1,23.26,5,371,...,23.26,5,0,23.26,0,0,0,0,0,1
4,ac371b31-b903-41bc-98b6-184f1232d58f,BNLX_ChurnP_SKUe_test_export.csv,1,NaN,hbi|eu|nl,440,16,405.43,46,"1,441",...,197.53,20,0,197.53,7,151.13,18,0,151.13,1


In [3]:
def incentive_distribution_comparison(df: pd.DataFrame) -> pd.DataFrame:
    before = (
        df["original_incentive_name"]
        .value_counts(normalize=False)
        .rename("before")
    )
    
    after = (
        df.loc[df["in_correction"] == 1, "original_incentive_name"]
        .value_counts(normalize=False)
        .rename("after")
    )
    
    return (
        pd.concat([before, after], axis=1)
        .fillna(0)
        .reset_index()
        .rename(columns={"index": "original_incentive_name"})
        .sort_values("before", ascending=False)
    )


df_incentive_dist = incentive_distribution_comparison(df)

df_incentive_dist

,original_incentive_name,before,after
0,BNLX_ChurnP_5eu_test_export.csv,3403,2711
1,BNLX_ChurnP_500_test_export.csv,3305,2685
2,BNLX_ChurnP_10eu_test_export.csv,3072,2526
3,BNLX_ChurnP_250_test_export.csv,3012,2373
4,BNLX_ChurnP_25_test_export.csv,2767,2244
5,BNLX_ChurnP_SKUe_test_export.csv,2638,2116
6,BNLX_ChurnP_10_test_export.csv,2594,2065


In [4]:
def coerce_metrics_to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()

    df[cols] = (
        df[cols]
        .replace({",": ""}, regex=True)
        .apply(pd.to_numeric, errors="coerce")
    )
    return df

In [5]:
def compute_means_comparison(df: pd.DataFrame) -> pd.DataFrame:
    df = coerce_metrics_to_numeric(
        df,
        ["frequency", "monetary_value", "total_volume"]
    )
    
    metrics = ["frequency", "monetary_value", "total_volume"]
    output = []
    
    for metric in metrics:
        overall = (
            df
            .groupby("original_incentive_name")[metric]
            .mean()
            .rename("before")
        )
        
        filtered = (
            df.loc[df["in_correction"] == 1]
            .groupby("original_incentive_name")[metric]
            .mean()
            .rename("after")
        )
        
        combined = (
            pd.concat([overall, filtered], axis=1)
            .assign(metric=metric)
            .reset_index()
        )
        
        output.append(combined)
    
    df_incentive_level = pd.concat(output, ignore_index=True)
    
    # total (not split by incentive)
    total_rows = []
    
    for metric in metrics:
        total_before = df[metric].mean()
        total_after = df.loc[df["in_correction"] == 1, metric].mean()
        
        total_rows.append({
            "original_incentive_name": "TOTAL",
            "before": total_before,
            "after": total_after,
            "metric": metric
        })
    
    df_total = pd.DataFrame(total_rows)
    
    return pd.concat([df_incentive_level, df_total], ignore_index=True)


df_comparison = compute_means_comparison(df)

df_comparison

,original_incentive_name,before,after,metric
0,BNLX_ChurnP_10_test_export.csv,5.090979,4.953027,frequency
1,BNLX_ChurnP_10eu_test_export.csv,4.028646,4.000000,frequency
2,BNLX_ChurnP_250_test_export.csv,3.865538,3.830173,frequency
3,BNLX_ChurnP_25_test_export.csv,4.363571,4.343137,frequency
4,BNLX_ChurnP_500_test_export.csv,4.267474,4.139292,frequency
5,BNLX_ChurnP_5eu_test_export.csv,3.912430,3.805238,frequency
6,BNLX_ChurnP_SKUe_test_export.csv,4.970432,4.865312,frequency
7,BNLX_ChurnP_10_test_export.csv,107.729271,105.516780,monetary_value
8,BNLX_ChurnP_10eu_test_export.csv,81.179671,80.479541,monetary_value
9,BNLX_ChurnP_250_test_export.csv,83.758864,79.638782,monetary_value
